# 🏥 Aged Care Demand Forecasting
## Notebook 02: Real Data Collection — ABS + AIHW + GEN

> **All data is free, publicly available, and licensed CC BY 4.0**  
> Downloads real Excel/CSV files from ABS and AIHW, parses them,  
> and saves clean CSVs to `data/raw/` for use in notebooks 03–07.

---

## 1. Setup & Imports

In [ ]:
# Standard library
import os
import zipfile
import io
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Data handling
import pandas as pd
import numpy as np
import requests

# Display settings
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Libraries imported successfully")
print(f"pandas  version: {pd.__version__}")
print(f"numpy   version: {np.__version__}")
print(f"requests version: {requests.__version__}")

In [ ]:
# Paths — Path objects so both / operator and os.path.join work throughout
NOTEBOOK_DIR  = Path(os.path.abspath('__file__')).parent
PROJECT_ROOT  = NOTEBOOK_DIR.parent
RAW_DIR       = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# FIX: HEADERS defined here so download_file can reference it
HEADERS = {'User-Agent': 'Mozilla/5.0 (research project; aged-care-ds)'}

print('Project paths configured:')
print(f'  Project root : {PROJECT_ROOT}')
print(f'  Raw data     : {RAW_DIR}')
print(f'  Processed    : {PROCESSED_DIR}')

## 2. Download Helper Function

Rather than downloading files manually, we write a reusable function that:
- Checks if the file already exists before downloading (avoids re-downloading)
- Shows file size after download
- Handles network errors gracefully

In [ ]:
def download_file(url, dest_path, label=''):
    dest_path = Path(dest_path)
    if dest_path.exists():
        print(f'  Already downloaded: {dest_path.name}')
        return dest_path
    print(f'  Downloading {label or dest_path.name} ...')
    r = requests.get(url, headers=HEADERS, timeout=120)
    r.raise_for_status()
    dest_path.write_bytes(r.content)
    size_kb = dest_path.stat().st_size / 1024
    print(f'     Saved: {dest_path.name} ({size_kb:.0f} KB)')
    return dest_path

---
## 3. ABS SEIFA 2021 — SA2 Level

**Source:** ABS Cat. 2033.0.55.001 — SEIFA 2021  
**Manual download required:**
1. Go to: https://www.abs.gov.au/statistics/people/people-and-communities/socio-economic-indexes-areas-seifa-australia/latest-release
2. Click **Data downloads**
3. Download **Statistical Area Level 2, 2021** Excel file
4. Save as `data/raw/seifa_2021.xlsx`

In [ ]:
SEIFA_FILE = RAW_DIR / 'seifa_2021.xlsx'

if not SEIFA_FILE.exists():
    raise FileNotFoundError(
        f'seifa_2021.xlsx missing at: {SEIFA_FILE}\n'
        'Download from: https://www.abs.gov.au/statistics/people/people-and-communities/'
        'socio-economic-indexes-areas-seifa-australia/latest-release\n'
        'Save to: data/raw/seifa_2021.xlsx'
    )

try:
    xl = pd.ExcelFile(SEIFA_FILE)
    print(f'✅ Successfully connected to SEIFA file')
    print(f'Sheets in file: {xl.sheet_names}')
except Exception as e:
    print(f'❌ Error reading Excel file: {e}')

In [ ]:
def read_seifa_sheet(xl, sheet_name, index_name):
    # Use header=5 to get the actual column names (Score, Decile, etc.)
    df = xl.parse(sheet_name, header=5)

    # Drop empty rows (usually the footer notes)
    df = df.dropna(subset=[df.columns[0]])

    # Clean column names
    df.columns = df.columns.str.strip()

    col_map = {}
    for col in df.columns:
        cl = str(col).lower()
        if '9-digit code' in cl:                      col_map[col] = 'sa2_code'
        elif 'sa2) name' in cl:                       col_map[col] = 'sa2_name'
        elif 'state' in cl and 'rank' not in cl:      col_map[col] = 'state'
        elif col == 'Score':                          col_map[col] = f'{index_name}_score'
        elif col == 'Decile':                         col_map[col] = f'{index_name}_decile'
        elif col == 'Percentile':                     col_map[col] = f'{index_name}_percentile'

    df = df.rename(columns=col_map)
    keep = [c for c in ['sa2_code', 'sa2_name', 'state',
                        f'{index_name}_score', f'{index_name}_decile',
                        f'{index_name}_percentile'] if c in df.columns]
    return df[keep].copy()

# Table 2 = IRSD, Table 3 = IRSAD
irsd_df  = read_seifa_sheet(xl, 'Table 2', 'irsd')
irsad_df = read_seifa_sheet(xl, 'Table 3', 'irsad')

seifa_df = irsd_df.merge(
    irsad_df[['sa2_code', 'irsad_score', 'irsad_decile']], on='sa2_code', how='left'
)
seifa_df['sa2_code'] = (
    seifa_df['sa2_code'].astype(str).str.strip()
    .str.replace('.0', '', regex=False).str.zfill(9)
)
seifa_df.to_csv(RAW_DIR / 'seifa_2021.csv', index=False)

print(f"✅ Successfully processed {len(seifa_df)} regions.")
seifa_df.head()

---
## 4. ABS Remoteness Structure — SA2 to Remoteness Category

**Auto-downloaded** from ABS ASGS Edition 3 allocation files.

In [ ]:
REMOTENESS_URL = (
    'https://www.abs.gov.au/statistics/standards/'
    'australian-statistical-geography-standard-asgs-edition-3/'
    'jul2021-jun2026/access-and-downloads/allocation-files/'
    'RA_2021_AUST.xlsx'
)
REMOTENESS_FILE = RAW_DIR / 'remoteness_sa1_2021.xlsx'

try:
    download_file(REMOTENESS_URL, REMOTENESS_FILE, 'ABS Remoteness SA1 2021')
    remote_raw = pd.read_excel(REMOTENESS_FILE, sheet_name=0, header=0)
    remote_raw.columns = remote_raw.columns.str.strip().str.lower().str.replace(' ', '_')
    print(f'✅ Loaded remoteness data')
    print(f'Columns: {list(remote_raw.columns)}')
    remote_ok = True
except Exception as e:
    print(f'❌ Failed: {e}')
    remote_raw = None
    remote_ok = False

In [ ]:
if remote_ok and remote_raw is not None:
    cols = remote_raw.columns.tolist()
    sa1_candidates = [c for c in cols if c.startswith('sa1')]
    ra_candidates  = [c for c in cols if c.startswith('ra_code')]

    if not (sa1_candidates and ra_candidates):
        print(f'❌ Column detection failed. Available: {cols}')
    else:
        sa1_col = sa1_candidates[0]
        ra_col  = ra_candidates[0]

        df = remote_raw[[sa1_col, ra_col]].copy()
        df['sa2_code'] = df[sa1_col].astype(str).str.zfill(11).str[:9]

        remote_sa2 = (
            df.groupby('sa2_code')[ra_col]
            .agg(lambda x: x.mode()[0])
            .reset_index()
            .rename(columns={ra_col: 'remoteness_cat'})
        )

        # FIX: Remap ABS subcategory codes (10-59) to standard 1-5 ARIA scale
        # 10-19 = Major Cities → 1, 20-29 = Inner Regional → 2
        # 30-39 = Outer Regional → 3, 40-49 = Remote → 4, 50-59 = Very Remote → 5
        def map_remoteness(code):
            if 10 <= code <= 19: return 1
            if 20 <= code <= 29: return 2
            if 30 <= code <= 39: return 3
            if 40 <= code <= 49: return 4
            if 50 <= code <= 59: return 5
            return 1  # fallback
        remote_sa2['remoteness_cat'] = remote_sa2['remoteness_cat'].apply(map_remoteness)

        # Reload from disk to avoid stale in-memory state
        seifa_df = pd.read_csv(RAW_DIR / 'seifa_2021.csv', dtype={'sa2_code': str})
        seifa_df['sa2_code'] = seifa_df['sa2_code'].str.zfill(9)
        seifa_df = seifa_df.drop(columns=['remoteness_cat'], errors='ignore')
        seifa_df = seifa_df.merge(remote_sa2, on='sa2_code', how='left')
        seifa_df['remoteness_cat'] = seifa_df['remoteness_cat'].fillna(1).astype(int)
        seifa_df.to_csv(RAW_DIR / 'seifa_2021.csv', index=False)

        print(f'✅ Matched {remote_sa2.shape[0]} SA2s')
        print('Remoteness distribution (1=Major Cities … 5=Very Remote):')
        print(seifa_df['remoteness_cat'].value_counts().sort_index())

else:
    seifa_df = pd.read_csv(RAW_DIR / 'seifa_2021.csv', dtype={'sa2_code': str})
    seifa_df['remoteness_cat'] = 1
    seifa_df.to_csv(RAW_DIR / 'seifa_2021.csv', index=False)
    print('Remoteness defaulted to 1.')

---
## 5. AIHW GEN — People Using Aged Care by Region (SA3)

**Manual download required** — three separate files from:  
https://www.gen-agedcaredata.gov.au/resources/access-data/2026/february/gen-data-people-using-aged-care-by-region

| File | Care Type | Sheet to use |
|------|-----------|--------------|
| `GEN-data-...-1-home-support-(recipient-location).xlsx` | CHSP | Table 1.2 (SA3) |
| `GEN-data-...-2-home-care-(recipient-location).xlsx` | Home Care Packages | Table 2.2 (SA3) |
| `GEN-data-...-4-residential-care-(service-location).xlsx` | Residential Care | Table 4.2 (SA3) |

Save all three files directly to `data/raw/` with the original filenames.

In [ ]:
# ── §5. AIHW GEN — People Using Aged Care by Region (SA3) ─────────────────────
CHSP_FILE = RAW_DIR / 'GEN-data-People-using-aged-care-by-region-2024-25-1-home-support-(recipient-location).xlsx'
HCP_FILE  = RAW_DIR / 'GEN-data-People-using-aged-care-by-region-30-June-2025-2-home-care-(recipient-location).xlsx'
RES_FILE  = RAW_DIR / 'GEN-data-People-using-aged-care-by-region-30-June-2025-4-residential-care-(service-location).xlsx'

for fpath, label in [(CHSP_FILE, 'CHSP'), (HCP_FILE, 'HCP'), (RES_FILE, 'Residential')]:
    if not fpath.exists():
        raise FileNotFoundError(
            f'{label} file missing.\n'
            f'Download from: https://www.gen-agedcaredata.gov.au/resources/access-data/'
            f'2026/february/gen-data-people-using-aged-care-by-region\n'
            f'Save to: {fpath}'
        )

def parse_gen_sa3(fpath, sheet_name, recipient_col):
    """Parse a GEN aged care by region SA3 sheet into a clean DataFrame."""
    df = pd.read_excel(fpath, sheet_name=sheet_name, header=2)
    df.columns = ['sa3_code', 'sa3_name'] + list(df.columns[2:])
    df = df.dropna(subset=['sa3_code']).copy()
    df['sa3_code'] = (
        df['sa3_code'].astype(str).str.strip()
        .str.replace(r'\.0$', '', regex=True)
        .str.zfill(5)
    )
    df['sa3_name'] = df['sa3_name'].astype(str).str.strip()
    df[recipient_col] = pd.to_numeric(df['Total'], errors='coerce').fillna(0).astype(int)
    return df[['sa3_code', 'sa3_name', recipient_col]]

chsp_sa3 = parse_gen_sa3(CHSP_FILE, 'Table 1.2 (SA3)', 'chsp_recipients')
hcp_sa3  = parse_gen_sa3(HCP_FILE,  'Table 2.2 (SA3)', 'home_care_recipients')
res_sa3  = parse_gen_sa3(RES_FILE,  'Table 4.2 (SA3)', 'residential_recipients')

print(f'CHSP SA3 regions:        {len(chsp_sa3):>4}')
print(f'Home Care SA3 regions:   {len(hcp_sa3):>4}')
print(f'Residential SA3 regions: {len(res_sa3):>4}')

aihw_sa3 = (
    chsp_sa3
    .merge(hcp_sa3, on=['sa3_code', 'sa3_name'], how='outer')
    .merge(res_sa3, on=['sa3_code', 'sa3_name'], how='outer')
)
for col in ['chsp_recipients', 'home_care_recipients', 'residential_recipients']:
    aihw_sa3[col] = aihw_sa3[col].fillna(0).astype(int)

aihw_sa3['total_recipients'] = (
    aihw_sa3['chsp_recipients'] +
    aihw_sa3['home_care_recipients'] +
    aihw_sa3['residential_recipients']
)
aihw_sa3 = aihw_sa3.sort_values('sa3_code').reset_index(drop=True)
aihw_sa3.to_csv(RAW_DIR / 'aihw_recipients_sa3.csv', index=False)

print(f'\n✅ Merged SA3 regions:      {len(aihw_sa3):>4}')
print(f'Total CHSP recipients:        {aihw_sa3["chsp_recipients"].sum():>10,}')
print(f'Total Home Care recipients:   {aihw_sa3["home_care_recipients"].sum():>10,}')
print(f'Total Residential recipients: {aihw_sa3["residential_recipients"].sum():>10,}')
print(f'Total all recipients:         {aihw_sa3["total_recipients"].sum():>10,}')
aihw_sa3.head()

---
## 6. Map SA3 → SA2 Using ABS Geographic Hierarchy

AIHW data is at SA3. We use the ABS SA2→SA3 hierarchy to distribute recipients  
to SA2 level using **population-weighted distribution** (irsd_score as proxy  
until real 70+ population data is available from §7).

In [ ]:
HIER_URL  = (
    'https://www.abs.gov.au/statistics/standards/'
    'australian-statistical-geography-standard-asgs-edition-3/'
    'jul2021-jun2026/access-and-downloads/allocation-files/'
    'SA2_2021_AUST.xlsx'
)
HIER_FILE = RAW_DIR / 'abs_sa2_hierarchy_2021.xlsx'

try:
    download_file(HIER_URL, HIER_FILE, 'ABS SA2 Geographic Hierarchy 2021')
    hier_df = pd.read_excel(HIER_FILE, sheet_name=0, header=0)
    hier_df.columns = hier_df.columns.str.strip().str.lower().str.replace(' ', '_')
    print(f'✅ Hierarchy loaded: {hier_df.shape}')
    print(f'Columns: {list(hier_df.columns)}')
    hier_ok = True
except Exception as e:
    print(f'❌ Auto-download failed: {e}')
    hier_df = None
    hier_ok = False

In [ ]:
if hier_ok and hier_df is not None:
    sa2_col      = [c for c in hier_df.columns if 'sa2' in c and 'code' in c][0]
    sa3_col      = [c for c in hier_df.columns if 'sa3' in c and 'code' in c][0]
    sa2_name_col = [c for c in hier_df.columns if 'sa2' in c and 'name' in c][0]
    state_col    = [c for c in hier_df.columns if 'state' in c][0]

    sa2_to_sa3 = hier_df[[sa2_col, sa2_name_col, sa3_col, state_col]].copy()
    sa2_to_sa3.columns = ['sa2_code', 'sa2_name', 'sa3_code', 'state']
    sa2_to_sa3['sa2_code'] = sa2_to_sa3['sa2_code'].astype(str).str.zfill(9)
    sa2_to_sa3['sa3_code'] = sa2_to_sa3['sa3_code'].astype(str).str.zfill(5)
    sa2_to_sa3 = sa2_to_sa3.dropna()

    # Population-weighted distribution
    # TODO: change WEIGHT_COL to 'pop_70plus' after §7 completes
    WEIGHT_COL = 'irsd_score'

    seifa_pop = pd.read_csv(RAW_DIR / 'seifa_2021.csv', dtype={'sa2_code': str})
    seifa_pop['sa2_code'] = seifa_pop['sa2_code'].str.zfill(9)

    sa2_to_sa3 = sa2_to_sa3.merge(
        seifa_pop[['sa2_code', WEIGHT_COL]], on='sa2_code', how='left'
    )
    sa2_to_sa3[WEIGHT_COL] = sa2_to_sa3[WEIGHT_COL].fillna(0)

    sa3_weight_total = (
        sa2_to_sa3.groupby('sa3_code')[WEIGHT_COL]
        .sum().rename('sa3_weight_total')
    )
    sa2_to_sa3 = sa2_to_sa3.merge(sa3_weight_total, on='sa3_code', how='left')

    sa2_count = sa2_to_sa3.groupby('sa3_code')['sa2_code'].transform('count')
    sa2_to_sa3['weight'] = (
        sa2_to_sa3[WEIGHT_COL] / sa2_to_sa3['sa3_weight_total'].replace(0, np.nan)
    ).fillna(1 / sa2_count)

    sa2_to_sa3 = sa2_to_sa3.merge(
        aihw_sa3[['sa3_code', 'residential_recipients', 'home_care_recipients',
                  'chsp_recipients', 'total_recipients']],
        on='sa3_code', how='left'
    )
    for col in ['residential_recipients', 'home_care_recipients',
                'chsp_recipients', 'total_recipients']:
        sa2_to_sa3[col] = (
            sa2_to_sa3[col] * sa2_to_sa3['weight']
        ).round(0).fillna(0).astype(int)

    aihw_sa2 = sa2_to_sa3.drop(columns=[WEIGHT_COL, 'sa3_weight_total', 'weight'])
    aihw_sa2.to_csv(RAW_DIR / 'aihw_recipients_sa2.csv', index=False)

    print(f'✅ AIHW recipients at SA2 level: {len(aihw_sa2):,} regions')
    print(f'   Weight method: population-weighted by {WEIGHT_COL}')
    print(f'   Total recipients distributed: {aihw_sa2["total_recipients"].sum():,}')
    aihw_sa2.head()
else:
    print('❌ SA2->SA3 mapping unavailable. Recipients remain at SA3 level.')

---
## 7. ABS Population by Age — SA2 Level

**Auto-download attempted** from ABS Cat. 3235.0  
File: `32350DS0001_2024.xlsx` — Population estimates by age and sex, by SA2, 2001–2024

If auto-download fails (ABS blocks some requests), manual download:
1. Go to: https://www.abs.gov.au/statistics/people/population/regional-population-age-and-sex/latest-release
2. Under **Data downloads** → *Population estimates by age and sex, by SA2 and above*
3. Save as `data/raw/abs_population_sa2.xlsx`

In [ ]:
POP_URL  = (
    'https://www.abs.gov.au/statistics/people/population/'
    'regional-population-age-and-sex/2024/32350DS0001_2024.xlsx'
)
POP_FILE = RAW_DIR / 'abs_population_sa2.xlsx'

try:
    download_file(POP_URL, POP_FILE, 'ABS Regional Population by Age and Sex SA2 2024')
    xl_pop = pd.ExcelFile(POP_FILE)
    print(f'✅ ABS Population file loaded')
    print(f'Sheets: {xl_pop.sheet_names}')
    print()
    # Show first data sheet structure to confirm layout
    print('Preview (identify header row and columns):')
    display(xl_pop.parse(xl_pop.sheet_names[1], header=None, nrows=10))
    pop_ok = True
except Exception as e:
    print(f'❌ Auto-download failed: {e}')
    print('Manual download: https://www.abs.gov.au/statistics/people/population/'
          'regional-population-age-and-sex/latest-release')
    print(f'Save to: {POP_FILE}')
    xl_pop = None
    pop_ok = False

In [ ]:
def process_abs_population(xl_pop):
    """
    Auto-detect header row and SA2/age columns in the ABS population file.
    Works for Table 1 (Males), Table 2 (Females), Table 3 (Persons).
    We use Table 3 (Total Persons).
    """
    # Try Table 3 first (Persons), fall back to first data sheet
    target_sheet = 'Table 3' if 'Table 3' in xl_pop.sheet_names else xl_pop.sheet_names[1]
    print(f'Using sheet: {target_sheet}')

    # Scan first 15 rows to find the header row (contains 'SA2' or 'Code')
    raw = xl_pop.parse(target_sheet, header=None, nrows=15)
    header_row = 0
    for i, row in raw.iterrows():
        row_str = ' '.join(str(v).lower() for v in row.values)
        if 'sa2' in row_str or 'code' in row_str:
            header_row = i
            break
    print(f'Detected header row: {header_row}')

    df = xl_pop.parse(target_sheet, header=header_row)
    df.columns = [str(c).strip() for c in df.columns]

    # Drop fully empty rows and columns
    df = df.dropna(how='all').dropna(axis=1, how='all')

    # Find SA2 code and name columns
    code_col  = next((c for c in df.columns if 'code' in c.lower() and 'sa2' in c.lower()), None)
    name_col  = next((c for c in df.columns if 'name' in c.lower() and 'sa2' in c.lower()), None)

    # If not found by name, use positional (code=col0, name=col1 in ABS files)
    if code_col is None: code_col = df.columns[0]
    if name_col is None: name_col = df.columns[1]

    print(f'SA2 code col: {code_col}')
    print(f'SA2 name col: {name_col}')

    # Find age band columns — ABS uses integer labels (0, 5, 10 ... 85) or '85+'
    age_cols = []
    for c in df.columns:
        c_str = str(c).strip().replace('+', '').replace('-', '').replace(' ', '')
        try:
            val = int(float(c_str))
            if 0 <= val <= 85:
                age_cols.append(c)
        except:
            pass

    # Total column
    total_col = next((c for c in df.columns if 'total' in str(c).lower()), None)

    print(f'Age band columns found: {len(age_cols)} → {age_cols[:5]} ... {age_cols[-3:]}')
    print(f'Total column: {total_col}')

    # Keep only rows with valid SA2 codes (9-digit numeric)
    df = df.dropna(subset=[code_col]).copy()
    df[code_col] = (
        df[code_col].astype(str).str.strip()
        .str.replace(r'\.0$', '', regex=True).str.zfill(9)
    )
    df = df[df[code_col].str.match(r'^\d{9}$')]

    # Convert age columns to numeric
    for c in age_cols + ([total_col] if total_col else []):
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

    # Build output dataframe
    out = pd.DataFrame()
    out['sa2_code'] = df[code_col].values
    out['sa2_name'] = df[name_col].astype(str).str.strip().values

    # Sum 70+ age bands
    bands_70plus = [c for c in age_cols if int(float(str(c).replace('+','').strip())) >= 70]
    bands_65plus = [c for c in age_cols if int(float(str(c).replace('+','').strip())) >= 65]
    out['pop_70plus'] = df[bands_70plus].sum(axis=1).round(0).astype(int).values
    out['pop_65plus'] = df[bands_65plus].sum(axis=1).round(0).astype(int).values
    out['total_pop']  = (df[total_col].round(0).astype(int).values
                         if total_col else df[age_cols].sum(axis=1).round(0).astype(int).values)

    return out

if pop_ok and xl_pop is not None:
    pop_df = process_abs_population(xl_pop)
    print(f'\n✅ Processed {len(pop_df):,} SA2 regions')
    print(f'   Total pop_70plus: {pop_df["pop_70plus"].sum():,}')
    print(f'   Total pop_65plus: {pop_df["pop_65plus"].sum():,}')
    print(f'   Total population: {pop_df["total_pop"].sum():,}')
    pop_df.head()
else:
    print('❌ Population file not loaded — fix download step above first.')
    pop_df = None

In [ ]:
if pop_df is None:
    raise RuntimeError('Population data not loaded — fix Cell above first.')

# ABS Series B (medium growth) 65+ projected growth factors by state
# Source: ABS Population Projections, Australia (Cat. 3222.0), Series B
STATE_GROWTH_2031 = {
    'NSW': 1.285, 'VIC': 1.312, 'QLD': 1.358,
    'SA':  1.267, 'WA':  1.341, 'TAS': 1.243,
    'NT':  1.421, 'ACT': 1.389,
}
STATE_GROWTH_2041 = {
    'NSW': 1.521, 'VIC': 1.573, 'QLD': 1.642,
    'SA':  1.498, 'WA':  1.612, 'TAS': 1.461,
    'NT':  1.693, 'ACT': 1.654,
}

# Derive state from SA2 code first digit
STATE_MAP = {
    '1': 'NSW', '2': 'VIC', '3': 'QLD', '4': 'SA',
    '5': 'WA',  '6': 'TAS', '7': 'NT',  '8': 'ACT', '9': 'OT'
}
pop_df = pop_df.copy()
pop_df['state'] = pop_df['sa2_code'].str[0].map(STATE_MAP)

# Apply state-level growth rates to SA2-level base counts
pop_df['pop_70plus_2031'] = (
    pop_df['pop_70plus'] * pop_df['state'].map(STATE_GROWTH_2031).fillna(1.30)
).round(0).astype(int)

pop_df['pop_70plus_2041'] = (
    pop_df['pop_70plus'] * pop_df['state'].map(STATE_GROWTH_2041).fillna(1.55)
).round(0).astype(int)

pop_df['pct_70plus_2024'] = (
    pop_df['pop_70plus'] / pop_df['total_pop'].replace(0, np.nan)
).fillna(0).round(4)

pop_df['growth_rate_70plus_2031'] = (
    pop_df['state'].map(STATE_GROWTH_2031).fillna(1.30) - 1
).round(4)

# FIX: Save to RAW_DIR (not PROCESSED_DIR) so inventory check finds it
pop_df.to_csv(RAW_DIR / 'abs_population_projections.csv', index=False)

print(f'✅ Population projections saved: {len(pop_df):,} SA2 regions')
print(f'   Columns: {list(pop_df.columns)}')
pop_df.head()

In [ ]:
# ── Update SA2 recipient distribution with real 70+ population weights ─────────
# Now that we have real pop_70plus data, rerun the SA3→SA2 distribution
# using pop_70plus instead of irsd_score as the weight
if hier_ok and hier_df is not None and pop_df is not None:
    WEIGHT_COL = 'pop_70plus'

    sa2_to_sa3 = hier_df[[
        [c for c in hier_df.columns if 'sa2' in c and 'code' in c][0],
        [c for c in hier_df.columns if 'sa2' in c and 'name' in c][0],
        [c for c in hier_df.columns if 'sa3' in c and 'code' in c][0],
        [c for c in hier_df.columns if 'state' in c][0]
    ]].copy()
    sa2_to_sa3.columns = ['sa2_code', 'sa2_name', 'sa3_code', 'state']
    sa2_to_sa3['sa2_code'] = sa2_to_sa3['sa2_code'].astype(str).str.zfill(9)
    sa2_to_sa3['sa3_code'] = sa2_to_sa3['sa3_code'].astype(str).str.zfill(5)
    sa2_to_sa3 = sa2_to_sa3.dropna()

    sa2_to_sa3 = sa2_to_sa3.merge(
        pop_df[['sa2_code', WEIGHT_COL]], on='sa2_code', how='left'
    )
    sa2_to_sa3[WEIGHT_COL] = sa2_to_sa3[WEIGHT_COL].fillna(0)

    sa3_weight_total = (
        sa2_to_sa3.groupby('sa3_code')[WEIGHT_COL]
        .sum().rename('sa3_weight_total')
    )
    sa2_to_sa3 = sa2_to_sa3.merge(sa3_weight_total, on='sa3_code', how='left')

    sa2_count = sa2_to_sa3.groupby('sa3_code')['sa2_code'].transform('count')
    sa2_to_sa3['weight'] = (
        sa2_to_sa3[WEIGHT_COL] / sa2_to_sa3['sa3_weight_total'].replace(0, np.nan)
    ).fillna(1 / sa2_count)

    sa2_to_sa3 = sa2_to_sa3.merge(
        aihw_sa3[['sa3_code', 'residential_recipients', 'home_care_recipients',
                  'chsp_recipients', 'total_recipients']],
        on='sa3_code', how='left'
    )
    for col in ['residential_recipients', 'home_care_recipients',
                'chsp_recipients', 'total_recipients']:
        sa2_to_sa3[col] = (
            sa2_to_sa3[col] * sa2_to_sa3['weight']
        ).round(0).fillna(0).astype(int)

    aihw_sa2 = sa2_to_sa3.drop(columns=[WEIGHT_COL, 'sa3_weight_total', 'weight'])
    aihw_sa2.to_csv(RAW_DIR / 'aihw_recipients_sa2.csv', index=False)

    print(f'✅ AIHW SA2 recipients updated with real pop_70plus weights')
    print(f'   Regions: {len(aihw_sa2):,}')
    print(f'   Total recipients: {aihw_sa2["total_recipients"].sum():,}')
    aihw_sa2.head()
else:
    print('Skipped — hierarchy or population data not available.')

---
## 8. Aged Care Supply

Supply data is published at ACPR level only (no SA2 codes) and has heavy value suppression.  
Supply-side features will be derived in `04_feature_engineering.ipynb` from recipient counts  
and population data already collected above.

In [ ]:
# Supply features derived in 04_feature_engineering.ipynb:
#   - residential_beds_per_1000_elderly  = residential_recipients / pop_70plus * 1000
#   - home_care_rate_per_1000_elderly    = home_care_recipients   / pop_70plus * 1000
#   - chsp_rate_per_1000_elderly         = chsp_recipients        / pop_70plus * 1000
#   - service_gap_score = projected_70plus_growth - current_utilisation_rate
supply_ok = False
print('✅ Supply section complete — features derived in notebook 04.')

---
## 9. Data Inventory & Validation

In [ ]:
print('=== Raw Data Inventory ===')
expected = [
    ('seifa_2021.csv',                  'ABS SEIFA 2021 SA2 + Remoteness'),
    ('abs_population_sa2.xlsx',         'ABS Population by Age & Sex SA2'),
    ('abs_population_projections.csv',  'ABS Population Projections SA2'),
    ('aihw_recipients_sa3.csv',         'AIHW Recipients SA3'),
    ('aihw_recipients_sa2.csv',         'AIHW Recipients SA2 (distributed)'),
    ('remoteness_sa1_2021.xlsx',        'ABS Remoteness SA1 2021'),
    ('abs_sa2_hierarchy_2021.xlsx',     'ABS SA2 Geographic Hierarchy'),
]
all_ok = True
for fname, label in expected:
    fpath = RAW_DIR / fname
    if fpath.exists():
        kb = fpath.stat().st_size / 1024
        if fname.endswith('.csv'):
            df_check = pd.read_csv(fpath, dtype=str)
            rows = len(df_check)
        else:
            rows = '—'
        print(f'  ✅  {label:<48} {str(rows):>6} rows  ({kb:.1f} KB)')
    else:
        print(f'  ❌  MISSING  {label:<48} ← {fname}')
        all_ok = False

print()
if all_ok:
    print('✅ All data files present — ready for 03_eda.ipynb')
else:
    print('⚠️  Some files missing — check sections above.')
print()
print('➡️  Next: 03_eda.ipynb')